# Getting and Analyzing the Data

In [1]:
import requests
import pandas as pd
import json
import time
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import numpy as np

/Users/dianasolfonseca/Documents/drama_recommender/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Test of Kuryana a FastAPI‑based, serverless scraper for MyDramaList 

In [23]:
cloy = "https://kuryana.tbdh.app/search/q/crash%20landing%20on%20you"

response = requests.get(cloy)

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Error: {response.status_code}")

{'query': 'crash landing on you', 'results': {'dramas': [{'slug': '35729-emergency-lands-of-love', 'thumb': 'https://i.mydramalist.com/g0wylo_4s.jpg?v=1', 'mdl_id': 'mdl-35729', 'title': 'Crash Landing on You', 'ranking': '#71', 'type': 'Korean Drama', 'year': 2019, 'series': '16 episodes'}, {'slug': '60399-crash-landing-on-you-special-lunar-new-year', 'thumb': 'https://i.mydramalist.com/RkwOos.jpg?v=1', 'mdl_id': 'mdl-60399', 'title': 'Crash Landing on You Special: Lunar New Year', 'ranking': '#410', 'type': 'Korean Special', 'year': 2020, 'series': '1 episodes'}, {'slug': '784322-chef-s-unexpected-landing-hakka-kitchen', 'thumb': 'https://i.mydramalist.com/VXo6ZE_4s.jpg?v=1', 'mdl_id': 'mdl-784322', 'title': "Chef's Unexpected Landing: Hakka Kitchen", 'ranking': None, 'type': 'Taiwanese TV Show', 'year': 2024, 'series': '9 episodes'}, {'slug': '4880-godzilla-raids-again', 'thumb': 'https://i.mydramalist.com/aEgyms.jpg?v=1', 'mdl_id': 'mdl-4880', 'title': 'Godzilla Raids Again', 'rank

### My dramalist

In [ ]:
username = "14855658"
url = f"https://kuryana.tbdh.app/dramalist/{username}"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
else:
    print(f"Error: {response.status_code}")
    
    
drama_list = data["data"]["list"]
all_dramas = []

for status, content in drama_list.items():
    for item in content["items"]:
        item_data = {
            "name": item["name"],
            "id": item["id"],
            "score": float(item["score"]),
            "episode_seen": int(item["episode_seen"]),
            "episode_total": int(item["episode_total"]),
            "status": status
        }
        all_dramas.append(item_data)

df = pd.DataFrame(all_dramas)
print(df.head())

               name                               id  score  episode_seen  \
0    Our Generation               710277-oceans-time    0.0             0   
1            Reborn                   754391-huan-yu    8.5             0   
2     Revenged Love             793270-revenged-love   10.0             0   
3          18 Again             52941-eighteen-again    8.5            16   
4  A Killer Paradox  729303-the-murderer-and-the-toy    8.5             8   

   episode_total     status  
0             24   Watching  
1             23   Watching  
2             24   Watching  
3             16  Completed  
4              8  Completed  


#### Analysis of my watching preferences

In [40]:

# df.to_csv("../data/raw/drama_list_2025.csv", index=False)
df_mydramas = pd.read_csv("../data/raw/drama_list_2025.csv")

### Retrieve metadata from MyDramaList for dramas with at least 7 stars released since 2015

In [28]:
BASE_URL = "https://kuryana.tbdh.app"
YEARS = range(2015, 2026)
QUARTERS = [1, 2, 3, 4]
TARGET_COUNTRIES = ["South Korea", "China", "Japan"]
MIN_RATING = 7.0

all_dramas = []

for year in YEARS:
    for q in QUARTERS:
        url = f"{BASE_URL}/seasonal/{year}/{q}"
        print(f"Fetching: {url}")
        try:
            res = requests.get(url)
            res.raise_for_status()
            dramas = res.json()
            print(f"  ➤ {len(dramas)} dramas trouvés pour {year} Q{q}")
        except Exception as e:
            print(f"Erreur sur {year} Q{q} : {e}")
            continue

        for drama in dramas:
            country = drama.get("country")
            rating = drama.get("rating") or 0
            type = drama.get("type")
            content_type = drama.get("content_type")

            if (
                country in TARGET_COUNTRIES and
                isinstance(rating, (int, float)) and
                rating >= MIN_RATING and
                type == 'Drama' and
                content_type in ["Japanese Drama", "Korean Drama", "Chinese Drama"]
            ):
                all_dramas.append({
                    "id": drama.get("id"),
                    "title": drama.get("title"),
                    "year": year,
                    "quarter": q,
                    "country": country,
                    "type": type,
                    "content_type": content_type,
                    "rating": rating,
                    "ranking": drama.get("ranking"),
                    "popularity": drama.get("popularity"),
                    "genres": drama.get("genres"),
                    "synopsis": drama.get("synopsis"),
                    "url": f"https://mydramalist.com{drama.get('url')}"
                })
        time.sleep(0.5)

df = pd.DataFrame(all_dramas)
print(f"\n✅ Total dramas filtrés : {len(df)}")
df.to_csv("../data/raw/filtered_dramas_2015_plus.csv", index=False)

Fetching: https://kuryana.tbdh.app/seasonal/2015/1
  ➤ 619 dramas trouvés pour 2015 Q1
Fetching: https://kuryana.tbdh.app/seasonal/2015/2
  ➤ 607 dramas trouvés pour 2015 Q2
Fetching: https://kuryana.tbdh.app/seasonal/2015/3
  ➤ 730 dramas trouvés pour 2015 Q3
Fetching: https://kuryana.tbdh.app/seasonal/2015/4
  ➤ 751 dramas trouvés pour 2015 Q4
Fetching: https://kuryana.tbdh.app/seasonal/2016/1
  ➤ 791 dramas trouvés pour 2016 Q1
Fetching: https://kuryana.tbdh.app/seasonal/2016/2
  ➤ 794 dramas trouvés pour 2016 Q2
Fetching: https://kuryana.tbdh.app/seasonal/2016/3
  ➤ 846 dramas trouvés pour 2016 Q3
Fetching: https://kuryana.tbdh.app/seasonal/2016/4
  ➤ 950 dramas trouvés pour 2016 Q4
Fetching: https://kuryana.tbdh.app/seasonal/2017/1
  ➤ 913 dramas trouvés pour 2017 Q1
Fetching: https://kuryana.tbdh.app/seasonal/2017/2
  ➤ 859 dramas trouvés pour 2017 Q2
Fetching: https://kuryana.tbdh.app/seasonal/2017/3
  ➤ 864 dramas trouvés pour 2017 Q3
Fetching: https://kuryana.tbdh.app/seasonal

In [29]:

# df_filtered.to_csv("../data/raw/filtered_dramas_2015_plus.csv", index=False)
df_all_dramas = pd.read_csv("../data/raw/filtered_dramas_2015_plus.csv")

In [39]:
# Retrieve the list of the most popular dramas 
df['popularity'] = pd.to_numeric(df['popularity'], errors='coerce')
df = df.dropna(subset=['popularity'])
top_dramas = df.groupby('country', group_keys=False).apply(lambda x: x.nsmallest(100, 'popularity'), include_groups=False).sort_values('popularity')

In [32]:
top_dramas

,id,title,year,quarter,type,content_type,rating,ranking,popularity,genres,synopsis,url
5584,729705,Hidden Love,2023,2,Drama,Chinese Drama,9.0,65,61,"Comedy,Romance,Life,Youth","Sang Zhi falls in love with Duan Jia Xu, the b...",https://mydramalist.com/729705-hidden-love
2416,28723,The Untamed,2019,2,Drama,Chinese Drama,9.0,67,66,"Mystery,Wuxia,Fantasy","Wei Wu Xian and Lan Wang Ji, two talented disc...",https://mydramalist.com/28723-the-untamed
1789,23184,Meteor Garden,2018,3,Drama,Chinese Drama,7.9,2952,85,"Comedy,Romance,Youth,Drama","Shan Cai, an 18-year-old girl from a strugglin...",https://mydramalist.com/23184-meteor-garden
733,15031,Love O2O,2016,3,Drama,Chinese Drama,8.2,1279,87,"Business,Comedy,Romance,Drama","Xiao Nai is a gaming expert who, courtesy of h...",https://mydramalist.com/15031-love-o2o
3876,49241,Falling into Your Smile,2021,2,Drama,Chinese Drama,8.6,400,115,"Comedy,Romance,Youth,Sports","In the ultra-competitive world of e-sports, th...",https://mydramalist.com/49241-falling-into-you...
...,...,...,...,...,...,...,...,...,...,...,...,...
1832,28327,Thirty but Seventeen,2018,3,Drama,Korean Drama,8.4,695,119,"Music,Mystery,Comedy,Romance","Woo Seo Ri, a violin prodigy at seventeen who ...",https://mydramalist.com/28327-thirty-but-seven...
1233,22477,School 2017,2017,3,Drama,Korean Drama,8.1,1766,120,"Mystery,Comedy,Romance,Youth","The drama follows Ra Eun Ho, an 18-year-old gi...",https://mydramalist.com/22477-school-2017
7244,766179,When the Phone Rings,2024,4,Drama,Korean Drama,8.0,2262,121,"Thriller,Mystery,Romance,Drama","Baek Sa Eon, the youngest presidential spokesm...",https://mydramalist.com/766179-the-number-you-...
3618,58953,Mouse,2021,1,Drama,Korean Drama,8.9,116,123,"Thriller,Mystery,Psychological",A suspenseful story that asks the key question...,https://mydramalist.com/58953-mouse


In [36]:
top_ids = set(top_dramas['id'])
df['is_popular'] = df['id'].apply(lambda x: x in top_ids)

In [37]:
df[['title', 'country', 'popularity', 'is_popular']].head()

,title,country,popularity,is_popular
0,Test Content 2,Japan,99999,False
1,Hana Moyu,Japan,11255,False
2,Singles Villa,China,8377,False
3,Enchanting Neighbor,South Korea,13467,False
4,Hanayome no Ren Season 4,Japan,99999,False


In [ ]:
# df.to_csv("../data/processed/dramas_with_popularity_flag.csv", index=False)

#### Analysis on all dramas

## Testing Chroma Embeddings DB

In [4]:
from chromadb import PersistentClient

chroma_client = PersistentClient(path="../app/chroma_db")  # same folder
collection = chroma_client.get_or_create_collection(name="dramas")

print("Collection count:", collection.count())

Collection count: 7948


In [5]:
collection = chroma_client.get_or_create_collection(name="dramas")

results = collection.query(
    query_texts=["A romantic Korean fantasy drama with time travel"],
    n_results=5,
    where={"country": "South Korea"}
)
for doc in results['metadatas'][0]:
    print("🎬", doc['title'])
    print("🌍 Country:", doc['country'])
    print("⭐ Rating:", doc['rating'])
    print("📖 Synopsis:", doc.get('synopsis', 'No synopsis available'))
    print("-" * 80)

🎬 Mother of Mine
🌍 Country: South Korea
⭐ Rating: 7.7
📖 Synopsis: No synopsis available
--------------------------------------------------------------------------------
🎬 Queen for Seven Days
🌍 Country: South Korea
⭐ Rating: 8.2
📖 Synopsis: No synopsis available
--------------------------------------------------------------------------------
🎬 Dream Knight
🌍 Country: South Korea
⭐ Rating: 7.3
📖 Synopsis: No synopsis available
--------------------------------------------------------------------------------
🎬 TV Novel: My Mind’s Flower Rain
🌍 Country: South Korea
⭐ Rating: 7.1
📖 Synopsis: No synopsis available
--------------------------------------------------------------------------------
🎬 300 Year-Old Class of 2020
🌍 Country: South Korea
⭐ Rating: 7.2
📖 Synopsis: No synopsis available
--------------------------------------------------------------------------------


In [63]:
collection.count()

0